# v5控制模型跨年预测偏移归因

复用封存两个Ridge模型及原生Qlib特征，精确复现2020预测；用系数×标准化特征均值变化分解全2020与四季度相对2018/2019训练日的预测偏移。零新拟合、零候选特征、零账户；2021/2022不读标签值。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.core.recorder_sources import load_verified_recorder_objects
from src.workflow.minute_action_value import _source
root=Path.cwd();card=root/"research/stage1_v5_control_feature_shift_20261030"
plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
label_card=root/plan["label_source"]["card"]
assert sha(label_card/"summary.json")==plan["label_source"]["summary_sha256"]
control_card=root/plan["control_source"]["card"]
assert sha(control_card/"summary.json")==plan["control_source"]["summary_sha256"]
assert sha(control_card/"verification_manifest.json")==plan["control_source"]["manifest_sha256"]
assert json.loads((control_card/"nas_binding.json").read_text())["evidence"]["sha256"]==plan["control_source"]["archive_sha256"]
assert sha(label_card/"verification_manifest.json")==plan["label_source"]["manifest_sha256"]
assert json.loads((label_card/"nas_binding.json").read_text())["evidence"]["sha256"]==plan["label_source"]["nas_archive_sha256"]
assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
label_objects,label_receipt=load_verified_recorder_objects(label_card,expected_recorder_id=plan["label_source"]["recorder_id"],
    artifacts=["analysis/labels","analysis/audit"],expected_manifest_sha256=plan["label_source"]["manifest_sha256"])
labels=label_objects["analysis/labels"]
assert len(labels)==label_objects["analysis/audit"]["event_slots"]
labels=labels.loc[labels.trade_date.dt.year.le(2020) & labels.soxl_funded & labels.soxx_funded].copy()
source,source_receipt=_source(root,plan["training_source"],["analysis/native/dated_f696_0"])
frame_source,frame_receipt=_source(root,plan["frame_source"],["inputs/covered_frame"])
book=source["analysis/native/dated_f696_0"]
frame=frame_source["inputs/covered_frame"].reset_index(drop=True)
assert frame.trade_date.equals(book["authority"].trade_date)
frame=frame.loc[frame.trade_date.dt.year.le(2020)].copy()
assert list(frame.columns[3:42])==plan["base39_features"]
assert frame.feature_available_at.lt(frame.trade_date+pd.Timedelta(hours=9,minutes=46)).all()
state=frame[["trade_date",*plan["base39_features"],"soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]].copy()
state["risk_state"]=book["authority"].loc[book["authority"].trade_date.dt.year.le(2020),"risk_state"].to_numpy()
state["risk_clock"]=book["authority"].loc[book["authority"].trade_date.dt.year.le(2020),"risk_clock"].to_numpy()
quant=[]
for day in state.trade_date:
 stamp=pd.Timestamp(day)+pd.Timedelta(hours=9,minutes=47)
 pos=book["native_positions"].get(stamp)
 assert pos is not None
 quant.append([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ("SOXL","SOXX")])
state[["q_soxl","q_soxx"]]=np.asarray(quant,float)
paired=labels.merge(state,on="trade_date",how="left",validate="many_to_one")
assert paired[plan["base39_features"]].notna().all().all()
bars=D.features(["SOXL","SOXX"],["$close","$raw_volume","$price_valid"],start_time="2018-10-15 09:31:00",end_time="2020-12-31 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
 part=bars.xs(symbol,level="instrument").copy()
 part=part.loc[part.index.normalize().isin(pd.DatetimeIndex(paired.trade_date.unique()))]
 assert part["$price_valid"].eq(1).all()
 days=part.index.normalize();logp=np.log(part["$close"].astype(float))
 out=pd.DataFrame(index=part.index)
 for width in (5,15):out[f"minute_{symbol.lower()}_ret{width}"]=logp-logp.groupby(days).shift(width)
 out[f"minute_{symbol.lower()}_log1p_raw_volume"]=np.log1p(part["$raw_volume"].astype(float))
 out[f"{symbol.lower()}_mark"]=part["$close"].astype(float)
 parts.append(out)
minute=parts[0].join(parts[1],how="inner")
paired=paired.join(minute,on="bar_end_at",how="left")
assert paired[["soxl_mark","soxx_mark"]].notna().all().all()
paired["decision_soxl_weight"]=paired.q_soxl*paired.soxl_mark/paired.decision_nav
paired["decision_soxx_weight"]=paired.q_soxx*paired.soxx_mark/paired.decision_nav
paired["clock_fraction"]=(paired.bar_end_at.dt.hour*60+paired.bar_end_at.dt.minute-(9*60+30))/390.
assert paired.label_available_at.gt(paired.decision_at).all()
assert paired.trade_date.dt.year.isin([2018,2019,2020]).all()
state_features=plan["base39_features"]+plan["state_features"]
full_features=state_features+plan["ordinary_minute_features"]
assert len(set(full_features))==len(full_features)
packed=pd.concat({"feature":paired[full_features],"label":paired[[plan["primary_label"]]],
    "metadata":paired[["trade_date","decision_at","label_available_at"]]},axis=1)
packed.index=pd.MultiIndex.from_arrays([paired.decision_at,["SOXL_SOXX_CASH"]*len(paired)],names=["datetime","instrument"])
dataset=DatasetH(handler=DataHandlerLP.from_df(packed),segments={"train":("2018-10-15","2019-12-31"),"test":("2020-01-01","2020-12-31")})
train=dataset.prepare("train",col_set=["feature","label","metadata"])
test=dataset.prepare("test",col_set=["feature","label","metadata"])
first_test=pd.Timestamp(test.index.get_level_values("datetime").min())
train=train.loc[pd.to_datetime(train[("metadata","label_available_at")]).lt(first_test)].copy()
assert train.index.get_level_values("datetime").max()<first_test
assert train[("metadata","trade_date")].dt.year.isin([2018,2019]).all()
assert test[("metadata","trade_date")].dt.year.eq(2020).all()
ytrain=train[("label",plan["primary_label"])].to_numpy(float)
ytest=test[("label",plan["primary_label"])].to_numpy(float)
train_days=pd.to_datetime(train[("metadata","trade_date")]).to_numpy()
test_days=pd.to_datetime(test[("metadata","trade_date")]).to_numpy()
train_n=pd.Series(train_days).map(pd.Series(train_days).value_counts()).to_numpy(float)
weights=1./train_n

control_objects,control_receipt=load_verified_recorder_objects(control_card,
    expected_recorder_id=plan["control_source"]["recorder_id"],
    artifacts=["models/state_only","models/ordinary_minute","analysis/predictions"],
    expected_manifest_sha256=plan["control_source"]["manifest_sha256"])
saved=control_objects["analysis/predictions"].reset_index(drop=True)
assert len(saved)==len(test) and pd.to_datetime(saved.decision_at).equals(pd.DatetimeIndex(test.index.get_level_values("datetime")).to_series(index=range(len(test))))
assert np.allclose(saved.actual_value.to_numpy(float),ytest,rtol=0,atol=1e-14)
rows=[];summary=[];top={}
for name,features in (("state_only",state_features),("ordinary_minute",full_features)):
    model=control_objects["models/"+name]
    imp=model.named_steps["simpleimputer"];scale=model.named_steps["standardscaler"];ridge=model.named_steps["ridge"]
    xtr=train.loc[:,pd.IndexSlice["feature",features]]
    xte=test.loc[:,pd.IndexSlice["feature",features]]
    ztr=scale.transform(imp.transform(xtr));zte=scale.transform(imp.transform(xte))
    pred=ridge.predict(zte)
    saved_pred=saved["state_prediction" if name=="state_only" else "ordinary_prediction"].to_numpy(float)
    parity=float(np.max(np.abs(pred-saved_pred)))
    assert parity<=1e-12
    names=imp.get_feature_names_out(features)
    assert len(names)==len(ridge.coef_)==zte.shape[1]
    train_mean=np.average(ztr,axis=0,weights=weights)
    train_pred=float(np.average(ridge.predict(ztr),weights=weights))
    for period,index in [("2020",np.arange(len(test))),
                         *[(q,np.flatnonzero(pd.PeriodIndex(pd.DatetimeIndex(test_days),freq="Q").astype(str)==q))
                           for q in sorted(set(pd.PeriodIndex(pd.DatetimeIndex(test_days),freq="Q").astype(str)))]]:
        days=test_days[index];n=pd.Series(days).map(pd.Series(days).value_counts()).to_numpy(float);w=1/n
        mean=np.average(zte[index],axis=0,weights=w)
        contributions=ridge.coef_*(mean-train_mean)
        predicted=float(np.average(pred[index],weights=w));actual=float(np.average(ytest[index],weights=w))
        assert abs(float(contributions.sum())-(predicted-train_pred))<1e-12
        summary.append({"model":name,"period":period,"rows":len(index),"days":int(pd.Series(days).nunique()),
            "train_prediction_mean":train_pred,"period_prediction_mean":predicted,"period_actual_mean":actual,
            "prediction_shift":predicted-train_pred,"contribution_sum":float(contributions.sum()),"saved_prediction_max_error":parity})
        for feature,coefficient,a,b,contribution in zip(names,ridge.coef_,train_mean,mean,contributions):
            rows.append({"model":name,"period":period,"feature":str(feature),"coefficient":float(coefficient),
                         "train_scaled_mean":float(a),"period_scaled_mean":float(b),"prediction_shift_contribution":float(contribution)})
    chosen=[x for x in rows if x["model"]==name and x["period"]=="2020"]
    top[name]=sorted(chosen,key=lambda x:abs(x["prediction_shift_contribution"]),reverse=True)[:10]
shift=pd.DataFrame(rows);quarter=pd.DataFrame(summary)
shift.to_csv(card/"feature_shift.csv",index=False)
quarter.to_csv(card/"period_prediction_shift.csv",index=False)
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),
       "decision":"read_only_control_shift_attribution_no_candidate_feature",
       "source_readback_entries":{"labels":len(label_receipt),"control":len(control_receipt),"parent":len(source_receipt),"frame":len(frame_receipt)},
       "training_days":int(pd.Series(train_days).nunique()),"evaluation_days":int(pd.Series(test_days).nunique()),
       "full_year":quarter.loc[quarter.period.eq("2020")].to_dict("records"),"top_contributions":top,
       "new_model_fits":0,"new_native_accounts":0,"new_recorders":0,"candidate_feature_selected":False,
       "limits":["2020 outcomes are reused development feedback; feature attribution is diagnostic, not a tradable rule.",
                 "Funded-pair sample is conditioned on next-bar fill; not a live-intent policy.",
                 "2021/2022 label values were not evaluated."]}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"decision":audit["decision"],"full_year":audit["full_year"],"top":{k:[(x["feature"],x["prediction_shift_contribution"]) for x in v[:5]] for k,v in top.items()}},ensure_ascii=False))


## 中文结论

封存Ridge模型2020逐行预测复现最大误差0；特征系数×训练至2020标准化均值变化逐项之和与预测均值变化误差<1e-12。训练日均预测约−0.079bp，2020预测+0.291bp、真实−0.092bp；Q2预测+0.58bp、真实−0.12bp。alpha_probability、rv20、soxx_ret60、levered_decay_20和soxx_vol20贡献相互抵消，六个普通分钟字段净贡献约−0.007bp。此为固定控制跨年映射失准的描述性证据，不是因果特征、交易阈值或新信息门。2021/2022未用，零新拟合/账户，SOXX目标未过。全四季度与全部特征贡献在两个CSV中；下一步先用训练期内规则构造稳健控制，不能依据2020调旧Ridge。